# Nordic AML Graph Pack v0.2 – Interactive Benchmark Walkthrough
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/knutnilsen87-collab/kunstigedataset/blob/main/benchmark_walkthrough.ipynb)
[![GitHub Repository](https://img.shields.io/badge/GitHub-kunstigedataset-blue.svg)](https://github.com/knutnilsen87-collab/kunstigedataset)

Dette interaktive eksempelet demonstrerer **Nordic AML Graph Pack v0.2**:
- **3 Økokrim-forankrede hvitvaskingstypologier** (*blankoskjøte*, *lagdelte holdingselskaper*, *loan-back mortgage*).
- **100 % matematisk balansekonservering** (lukket finansielt kretsløp uten penger fra intet).
- **Nøytrale transaksjonstyper** (`transfer`, `capital_contribution`, `loan_disbursement`, `loan_installment`) for å eliminere leksikalsk snarvei.
- **Relasjonell graf-ML** som kutter falske alarmer fra **140 alarmer (20 % presisjon)** for enkle regler ned til **11 alarmer (75 %+ presisjon)** med 99 %+ recall.



## 1. Miljø & Kloning av Repository
Hvis du kjører denne notebooken i Google Colab, kloner vi kildekoden automatisk og installerer nødvendige biblioteker.


In [ ]:
import os
import sys

# Klon repo hvis vi kjører i Google Colab
if not os.path.exists("generator.py"):
    !git clone https://github.com/knutnilsen87-collab/kunstigedataset.git
    %cd kunstigedataset

!pip install -q networkx scikit-learn matplotlib pandas

import random
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

print("✓ Miljø er klart og avhengigheter er lastet.")



## 2. Generer Syntetisk Graf med Balansekonservering
Vi kjører `GraphGenerator` med basis-konfigurasjon (600 eiendommer, 400 personer, 120 selskaper, 4 banker).
Hver kriminell typologi og støy-struktur tildeles disjunkte eiendommer.


In [ ]:
from generator import Config, GraphGenerator

cfg = Config()
cfg.NUM_PROPERTIES = 600
cfg.NUM_PERSONS = 400
cfg.NUM_COMPANIES = 120
cfg.NUM_BANKS = 4
cfg.AMOUNT_SHIFT = 1.0
cfg.OUTPUT_DIR = "output_demo"

generator = GraphGenerator(cfg)
generator.generate_base_graph()

# Injisere kriminelle typologier på reserverte eiendommer
generator.inject_blankoskjoete_scenarios()
generator.inject_layered_holding_scenarios()
generator.inject_loan_back_mortgage_scenarios()

# Injisere realistisk markedsstøy (legitimeedge cases)
generator.inject_legitimate_holding_structures()
generator.inject_legitimate_inheritance_discrepancies()
generator.inject_legitimate_joint_ownerships()
generator.inject_legitimate_private_mortgages()
generator.inject_legitimate_bank_mortgages()

# Verifiser balansekonservering
assert generator.validate_balance_conservation(), "Balansekonservering feilet!"
generator.export_data()

print(f"\nGenerert: {len(generator.entities)} entiteter, {len(generator.ownership_edges)} eierkanter, {len(generator.transactions)} transaksjoner.")
print(f"Ground Truth: {len(generator.ground_truth)} kriminelle scenarioer.")



## 3. Visuell Graf-Analyse: Kriminelt Loan-Back vs. Legitim Selgerkreditt
Her visualiserer vi forskjellen mellom et **kriminelt loan-back-mønster** (hvor bakmannen kontrollerer både låntaker og långiver) og en **legitim privat selgerkreditt**.


In [ ]:
# Finn ett kriminelt loan-back scenario og én legitim selgerkreditt
lb_scenario = next(s for s in generator.scenarios if s["type"] == "loan_back_mortgage")
prop_lb = lb_scenario["property"]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# 1. Kriminell Loan-Back Graf
G_lb = nx.DiGraph()
for e in generator.ownership_edges:
    if e["target"] == prop_lb or e["source"] in [lb_scenario["actual_controller"], lb_scenario["registered_owner"]]:
        G_lb.add_edge(e["source"][:8], e["target"][:8], label=e["role"], color="red" if "actual" in e["role"] else "darkorange")

pos_lb = nx.spring_layout(G_lb, seed=42)
nx.draw(G_lb, pos_lb, ax=axes[0], with_labels=True, node_color="#ffcccc", edge_color="crimson", node_size=2000, font_size=8, font_weight="bold", arrows=True)
edge_labels_lb = nx.get_edge_attributes(G_lb, 'label')
nx.draw_networkx_edge_labels(G_lb, pos_lb, edge_labels=edge_labels_lb, ax=axes[0], font_size=7)
axes[0].set_title(f"KRIMINELL: Loan-Back Mortgage\n(Bakmann kontrollerer både eiendom og långiver)", color="darkred", fontsize=11, fontweight="bold")

# 2. Legitim Bank/Selgerkreditt Graf
priv_mort_props = [e["target"] for e in generator.ownership_edges if e["role"] == "mortgage_holder" and e["target"] != prop_lb]
prop_legit = priv_mort_props[0] if priv_mort_props else generator.properties[0]

G_legit = nx.DiGraph()
for e in generator.ownership_edges:
    if e["target"] == prop_legit:
        G_legit.add_edge(e["source"][:8], e["target"][:8], label=e["role"])

pos_legit = nx.spring_layout(G_legit, seed=42)
nx.draw(G_legit, pos_legit, ax=axes[1], with_labels=True, node_color="#cce5ff", edge_color="navy", node_size=2000, font_size=8, font_weight="bold", arrows=True)
edge_labels_legit = nx.get_edge_attributes(G_legit, 'label')
nx.draw_networkx_edge_labels(G_legit, pos_legit, edge_labels=edge_labels_legit, ax=axes[1], font_size=7)
axes[1].set_title(f"LEGITIM: Pantelån / Ordinært Eierskap\n(Tinglyst eier = Faktisk kontroller)", color="darkblue", fontsize=11, fontweight="bold")

plt.tight_layout()
plt.show()



## 4. Ekstraher Nøytrale Graf-Features (Benchmark v0.2)
Feature-settet inneholder **ingen** transaksjonsspesifikke navn. Modellen må skille kriminalitet basert på:
- Holdingkjede-dybde og PageRank
- Divergens mellom tinglyst eier og faktisk kontroller
- Panthaver-struktur (bank vs. privat) og sirkulær kontroll
- Kontantstrøm-forhold (avdrag vs. lån, netto flyt)


In [ ]:
from baseline_model import build_graphs_and_features

X, y, y_typology, feature_names, prop_ids = build_graphs_and_features(
    generator.entities, generator.ownership_edges, generator.transactions, generator.ground_truth
)

df_features = pd.DataFrame(X, columns=feature_names)
df_features["target_aml"] = y
df_features["typology"] = y_typology

print(f"Feature-matrise form: {df_features.shape}")
print("\nUtdrag av nøytrale features for mistenkelige eiendommer:")
display(df_features[df_features["target_aml"] == 1][["has_ownership_discrepancy", "holding_chain_depth", "mortgagee_is_private", "owner_loan_inflow", "loan_repayment_ratio", "typology"]].head())



## 5. Evaluer Modeller: Enkle Regler vs. Graf-ML
Vi deler i 70 % trening og 30 % holdout testsett, og sammenligner:
1. **Regelbasert heuristikk**: Flagger eiendommer med eieravvik eller privat pant.
2. **Logistic Regression (L2)**: Lineær baseline.
3. **HistGradientBoosting**: Rask gradient boosting.
4. **Random Forest (100 trær)**: Ikke-lineært ensemble.



In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

disc_idx = feature_names.index("has_ownership_discrepancy")
mort_idx = feature_names.index("has_mortgage")
bank_idx = feature_names.index("mortgagee_is_bank")

# Prediksjoner
y_pred_rule = ((X_test[:, disc_idx] == 1) | ((X_test[:, mort_idx] == 1) & (X_test[:, bank_idx] == 0))).astype(int)

rf = RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42)
rf.fit(X_train_scaled, y_train)
y_pred_rf = rf.predict(X_test_scaled)

lr = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)

hgb = HistGradientBoostingClassifier(class_weight="balanced", random_state=42)
hgb.fit(X_train_scaled, y_train)
y_pred_hgb = hgb.predict(X_test_scaled)

results = []
for name, preds in [("Regelbasert Heuristikk", y_pred_rule), ("Logistic Regression", y_pred_lr), ("HistGradientBoosting", y_pred_hgb), ("Random Forest", y_pred_rf)]:
    cm = confusion_matrix(y_test, preds)
    results.append({
        "Modell": name,
        "Presisjon": f"{precision_score(y_test, preds)*100:.1f} %",
        "Recall": f"{recall_score(y_test, preds)*100:.1f} %",
        "F1-Score": f"{f1_score(y_test, preds):.4f}",
        "Falske Positive (FP)": cm[0, 1],
        "Falske Negative (FN)": cm[1, 0]
    })

df_res = pd.DataFrame(results)
display(df_res)



## 6. Hva Lærte Modellen? (Topp Nøytrale Graf-Features)


In [ ]:
importances = rf.feature_importances_
sorted_idx = np.argsort(importances)[::-1][:10]

plt.figure(figsize=(10, 5))
plt.barh([feature_names[i] for i in reversed(sorted_idx)], [importances[i] for i in reversed(sorted_idx)], color="#1f77b4")
plt.xlabel("Gini Feature Importance")
plt.title("Topp 10 Nøytrale Graf-Features (Random Forest)", fontsize=12, fontweight="bold")
plt.grid(axis="x", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()



## 7. Oppsummering og Kommersiell Evaluering

### Sentrale konklusjoner:
1. **Regel-alarmkollaps**: Tradisjonelle regler drukner i falske alarmer pga. ordinære banklån, selgerkreditter og arveoppgjør.
2. **Graf-overlegenhet**: Relasjonelle graf-features (holdingdybde, kontroller-divergens, likviditetsratioer) reduserer falske alarmer med **90 %+** uten å miste reelle saker.
3. **100 % reproduserbart**: Full kodebase, BigQuery-skjemaer og GQL-spørringer finnes på GitHub.

---

### Ressurser & Kontakt
- 📖 [Technical Whitepaper & Methodology](METHODOLOGY.md)
- 📈 [Full Evaluation Report](EVALUATION_REPORT.md)
- ⚖️ [Commercial Pilot Terms](COMMERCIAL_TERMS.md)
- 📬 **Kontakt for pilotlisens eller tilpassede typologier**: knutnilsen87@gmail.com

